# 03 - Redes Neurais: Como o Cérebro Inspira IA

## Pergunta 27: Como redes neurais artificiais se inspiram no cérebro?

### Parte 3: Forward Pass, Backpropagation e Aprendizado

Este notebook conecta tudo:
- Como múltiplos neurônios formam redes
- Forward pass: propagação de sinais (como cérebro funciona)
- Backpropagation: ajuste de pesos (como cérebro aprende)
- Aplicação em diagnóstico de plantas

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle, FancyBboxPatch
from matplotlib.patches import ConnectionPatch

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Setup concluído")

## 1. Forward Pass: Como Sinais Propagam

In [ ]:
print("\n" + "="*70)
print("➡️ FORWARD PASS: Propagação de Sinais (Bio e IA)")
print("="*70)

print("""
NO CÉREBRO BIOLÓGICO:
  1. Neurônios sensoriais recebem estímulo (imagem de planta)
  2. Disparam e enviam sinais para próxima camada
  3. Próxima camada recebe, integra, dispara
  4. Processo se repete através de múltiplas camadas
  5. Camadas finais geram resposta (diagnóstico)
  
NA RNA ARTIFICIAL:
  1. Input (imagem pixels: 224×224 = 50.176 números)
  2. Camada 1: multiplica por pesos W1, soma bias b1 → z1 = W1×x + b1
  3. Ativação: a1 = ReLU(z1)
  4. Camada 2: a2 = ReLU(W2×a1 + b2)
  5. ...
  6. Output: ŷ = Softmax(Wn×a(n-1) + bn)
  
📊 PARALELO:
  ✓ Bio: neurônio → neurônio → ... → resposta
  ✓ IA:  input → camada1 → camada2 → ... → output
  
✨ RESULTADO:
  Bio: "É ferrugem" (diagnóstico)
  IA:  [0.0, 0.94, 0.03, 0.02, 0.01] → argmax = classe 1 (ferrugem)
""")

print("="*70)

## 2. Visualizar Forward Pass

In [ ]:
# Diagrama de forward pass
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# INPUT
ax.text(1, 7.5, 'INPUT\n(Imagem)', fontsize=11, fontweight='bold', ha='center',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
for i in range(4):
    ax.add_patch(Circle((1, 6 - i*1), 0.2, color='blue', alpha=0.7))
ax.text(1, 1.5, '4096 pixels\n(64×64)', fontsize=9, ha='center')

# SETAS COM PESOS
arrow1 = FancyArrowPatch((1.2, 5), (2.8, 5), arrowstyle='->', 
                        mutation_scale=20, linewidth=2, color='gray')
ax.add_patch(arrow1)
ax.text(2, 5.5, 'W1\n(4096×512)', fontsize=8, ha='center', style='italic')

# HIDDEN 1
ax.text(3.5, 7.5, 'HIDDEN 1\n512 neurons', fontsize=10, fontweight='bold', ha='center',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
for i in range(4):
    ax.add_patch(Circle((3.5, 6 - i*1), 0.15, color='green', alpha=0.7))
ax.text(3.5, 1.5, 'z1 = W1×x + b1\na1 = ReLU(z1)', fontsize=9, ha='center')

# SETAS
arrow2 = FancyArrowPatch((3.65, 5), (5.35, 5), arrowstyle='->', 
                        mutation_scale=20, linewidth=2, color='gray')
ax.add_patch(arrow2)
ax.text(4.5, 5.5, 'W2\n(512×256)', fontsize=8, ha='center', style='italic')

# HIDDEN 2
ax.text(6, 7.5, 'HIDDEN 2\n256 neurons', fontsize=10, fontweight='bold', ha='center',
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
for i in range(4):
    ax.add_patch(Circle((6, 6 - i*1), 0.12, color='orange', alpha=0.7))
ax.text(6, 1.5, 'z2 = W2×a1 + b2\na2 = ReLU(z2)', fontsize=9, ha='center')

# SETAS
arrow3 = FancyArrowPatch((6.12, 5), (7.88, 5), arrowstyle='->', 
                        mutation_scale=20, linewidth=2, color='gray')
ax.add_patch(arrow3)
ax.text(7, 5.5, 'W3\n(256×6)', fontsize=8, ha='center', style='italic')

# OUTPUT
ax.text(8.5, 7.5, 'OUTPUT\n6 classes', fontsize=10, fontweight='bold', ha='center',
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))
classes = ['Saudável', 'Ferrugem', 'Cercospora', 'Phoma', 'Bicho', 'Outro']
for i, cls in enumerate(classes):
    ax.add_patch(Circle((8.5, 5.5 - i*0.8), 0.15, color='red', alpha=0.7))
    ax.text(9.2, 5.5 - i*0.8, f'{cls}\n({0.94 if i==1 else 0.02:.2f})', 
           fontsize=8, va='center')

ax.text(8.5, 0.8, 'ŷ = Softmax(W3×a2 + b3)\nargmax = classe 1 (Ferrugem: 94%)', 
       fontsize=9, ha='center', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# TEMPO
ax.annotate('', xy=(11, 6.5), xytext=(1, 6.5),
           arrowprops=dict(arrowstyle='<->', lw=2, color='purple'))
ax.text(6, 6.8, '~1-2 ms (muito rápido!)', fontsize=10, ha='center', 
       bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.8), fontweight='bold')

ax.set_title('Forward Pass: De Imagem para Diagnóstico', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('results/plots/forward_pass_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Diagrama criado: forward_pass_visualization.png")

## 3. Backpropagation: Como Aprende (STDP Artificial)

In [ ]:
print("\n" + "="*70)
print("⬅️ BACKPROPAGATION: Aprendizado (Como Cérebro Aprende)")
print("="*70)

print("""
NO CÉREBRO (Plasticidade Sináptica / STDP):
  1. Neurônio A e B disparam juntos (correlação)
  2. Sinapse entre A→B se FORTALECE
  3. Próxima vez: A dispara → B dispara mais facilmente
  4. Resultado: padrão aprendido!
  
NA RNA ARTIFICIAL (Backpropagation):
  1. Treino: mostrar imagem com label correto
  2. Forward pass: rede faz predição (pode estar errada)
  3. Calcular erro: |ŷ - y_verdadeiro|
  4. BACKPROP: propagar erro para trás
     - Camada output: quanto cada peso W3 contribuiu pro erro?
     - Camada hidden2: quanto cada W2 contribuiu?
     - Camada hidden1: quanto cada W1 contribuiu?
  5. Atualizar pesos: w_novo = w_velho - α × gradiente
  6. RESULTADO: pesos ajustam para reduzir erro!

📊 O PARALELO:
  Bio STDP: "Use it or lose it" (sinapse usada = fortalecida)
  IA Backprop: "Ajuste os pesos na direção que reduz erro"
  
✨ AMBOS aprendem ajustando conexões com EXPERIÊNCIA!
""")

print("="*70)

## 4. Exemplo Concreto: Diagnóstico de Ferrugem

In [ ]:
print("\n" + "="*70)
print("🌾 EXEMPLO PRÁTICO: Diagnosticar Ferrugem na Soja")
print("="*70)

print("""
ITERACION 1 (Sem treinamento):
  Input: Imagem de folha com ferrugem
  Forward pass:
    • Neurônios sensoriais detectam padrões
    • Cada camada processa
    • Output: [0.2, 0.2, 0.2, 0.2, 0.1, 0.1] (completamente confuso!)
  
  Verdade: ferrugem = classe 1
  Predição: classe 2 (Cercospora) com 20% confiança
  Erro: ALTO! Predição errada
  
  Backprop: ajusta pesos para reconhecer ferrugem melhor

ITERACION 100:
  Input: Mesma imagem
  Forward pass:
    • Pesos já têm padrão de ferrugem
    • Neurônios ativam em cascata
    • Output: [0.01, 0.94, 0.02, 0.01, 0.01, 0.01]
  
  Verdade: ferrugem = classe 1
  Predição: classe 1 (Ferrugem) com 94% confiança
  Erro: BAIXO! Acertou!
  
  Backprop: pequeno ajuste (já está bom)

ITERACION 1000 (muitos exemplos):
  Input: NOVA imagem de ferrugem (nunca viu antes)
  Forward pass:
    • Pesos generalizaram o padrão
    • Reconhece ferrugem mesmo diferente
    • Output: [0.02, 0.91, 0.03, 0.01, 0.01, 0.02]
  
  Predição: Ferrugem com 91% confiança
  ✓ FUNCIONOU! Aprendeu!
""")

print("\n💡 POR QUE FUNCIONA?")
print("""
1. Recompensa positiva (erro baixo) → pesos que geraram saída correta se fortalecem
2. Punição negativa (erro alto) → pesos que geraram saída errada se enfraquecem
3. Depois de muitos exemplos → rede "aprendeu" o padrão de ferrugem
4. Quando vê nova imagem → ativa os mesmos neurônios → predição correta!

🧠 CONEXÃO COM CÉREBRO:
  • Você vê MUITAS maçãs vermelhas
  • Sinapses que detectam "vermelho + fruta" se fortalecem
  • Quando vê maçã nova → sinapses já estão preparadas
  • Reconhece automaticamente!

🤖 REDE NEURAL FAZ EXATAMENTE ISSO:
  • Treina em MUITAS imagens de ferrugem
  • Pesos que detectam "padrão de ferrugem" se fortalecem
  • Quando vê ferrugem nova → ativa os neurônios certos
  • Reconhece automaticamente!
""")

print("="*70)

## 5. Resumo Final: Como RNA Inspira-se no Cérebro

In [ ]:
print("\n" + "="*70)
print("🎯 RESPOSTA PARA P27: Como RNA se Inspira no Cérebro?")
print("="*70)

print("""
1️⃣ ESTRUTURA:
   ✓ Cérebro: múltiplos neurônios conectados
   ✓ RNA: múltiplos neurônios artificiais em camadas
   → Arquitetura similar

2️⃣ COMUNICAÇÃO:
   ✓ Cérebro: neurônios enviam sinais uns aos outros
   ✓ RNA: forward pass propaga ativações através das camadas
   → Processo similar (mas muito mais rápido em IA)

3️⃣ FORÇA DE CONEXÃO:
   ✓ Cérebro: sinapses tem força (plasticidade)
   ✓ RNA: neurônios tem pesos (w)
   → Mecanismo de "memória" similar

4️⃣ APRENDIZADO:
   ✓ Cérebro: plasticidade sináptica (STDP) - "neurons that fire together, wire together"
   ✓ RNA: backpropagation - ajusta pesos para reduzir erro
   → AMBOS aprendem mudando força das conexões

5️⃣ GENERALIZAÇÃO:
   ✓ Cérebro: depois de ver muitas maçãs, reconhece maçã nova
   ✓ RNA: depois de treinar em muitas imagens, reconhece imagem nova
   → Mesmo princípio: aprender padrões, não memorizar

6️⃣ NÃO-LINEARIDADE:
   ✓ Cérebro: tudo-ou-nada (dispara ou não)
   ✓ RNA: função de ativação (ReLU, Sigmoid, etc)
   → Ambos precisam de não-linearidade para complexidade

📌 CONCLUSÃO FINAL:

  RNA Artificial é uma IMITAÇÃO do cérebro que:
    ✓ Captura a essência (múltiplos neurônios → padrões complexos)
    ✓ Usa o mesmo princípio de aprendizado (ajustar força de conexões)
    ✓ Funciona surpreendentemente bem!
    
  MAS:
    ✗ Perde propriedades do cérebro real (robustez, eficiência, rapidez de aprendizado)
    ✗ Requer muito mais dados
    ✗ Consome muito mais energia
    ✗ É frágil a erros nos dados (próxima pergunta: P23)
""")

print("\n" + "="*70)
print("PRÓXIMA FASE: P23 - Como erros nos dados afetam o aprendizado?")
print("="*70)